# F1 Winner Prediction - Notebook 01: Feature Engineering & Dataset

Ejecuta el pipeline de feature engineering y construye las secuencias de entrenamiento.
Importa todo desde `src/`.

Output: features_train.pt (192 seqs), features_val.pt (33 seqs), features_2025.pt (24 seqs), metadata.pkl

In [ ]:
# @title 1. Clone Repo & Setup
!git clone https://github.com/USERNAME/f1_transformer.git 2>/dev/null || echo 'Repo already cloned'
%cd f1_transformer
!pip install -q -r requirements.txt 2>/dev/null

from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['COLAB'] = '1'

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import pickle
import numpy as np
print('Setup complete!')

In [ ]:
# @title 2. Run Build Dataset Pipeline
# This runs src/preprocessing/build_dataset.py which:
# 1. Loads raw race data
# 2. Computes driver form, constructor form, track history, championship position
# 3. Merges circuit & weather features
# 4. Builds encoders (driver, constructor, circuit)
# 5. Builds sequences (context=10 races, 20 drivers per race)
# 6. Normalizes features
# 7. Saves features_train.pt, features_val.pt, features_2025.pt, metadata.pkl

from src.preprocessing.build_dataset import main
main()

In [ ]:
# @title 3. Verify Dataset
PROCESSED = Path('/content/drive/MyDrive/f1_transformer/data/processed')

train_data = torch.load(PROCESSED / 'features_train.pt', weights_only=False)
val_data = torch.load(PROCESSED / 'features_val.pt', weights_only=False)
data_2025 = torch.load(PROCESSED / 'features_2025.pt', weights_only=False)

with open(PROCESSED / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

print('='*50)
print('DATASET VERIFICATION')
print('='*50)
print(f'Training sequences:   {len(train_data["winners"])}')
print(f'Validation sequences: {len(val_data["winners"])}')
print(f'2025 sequences:       {len(data_2025["winners"])}')
print(f'')
print(f'Candidate features:   {metadata["d_candidate_raw"]}')
print(f'  = 3 embeddings (driver+constructor+circuit) + numeric features')
print(f'Context features:     {metadata["d_context_raw"]}')
print(f'  = pooled driver stats + circuit + weather + top3 + winner')
print(f'')
print(f'Context window:       {metadata["context_window"]} races')
print(f'Drivers:              {metadata["num_drivers"]}')
print(f'Constructors:         {metadata["num_constructors"]}')
print(f'Circuits:             {metadata["num_circuits"]}')
print(f'')
print(f'Shapes:')
print(f'  Context:    {train_data["context"].shape}')
print(f'  Candidates: {train_data["candidates"].shape}')
print(f'  Winners:    {train_data["winners"].shape}')
print(f'  Time gaps:  {train_data["time_gaps"].shape}')
print('='*50)
print('\nReady for Notebook 02: Model Training!')